In [ ]:
# ==============================
# 1. Import Libraries
# ==============================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_error, mean_squared_error


# ==============================
# 2. Load Dataset
# ==============================
from statsmodels.datasets import airpassengers

data = airpassengers.load_pandas().data
data['Month'] = pd.to_datetime(data['Month'])
data.set_index('Month', inplace=True)

series = data['AirPassengers']

# Plot original series
plt.figure()
plt.plot(series)
plt.title("Original Time Series")
plt.show()


# ==============================
# 3. Check Stationarity (ADF Test)
# ==============================
result = adfuller(series)

print("ADF Statistic:", result[0])
print("p-value:", result[1])

# If p-value > 0.05 → not stationary


# ==============================
# 4. Apply First Differencing (d=1)
# ==============================
series_diff = series.diff().dropna()

plt.figure()
plt.plot(series_diff)
plt.title("First Differenced Series")
plt.show()

result_diff = adfuller(series_diff)

print("ADF After Differencing:", result_diff[0])
print("p-value After Differencing:", result_diff[1])


# ==============================
# 5. Train-Test Split
# ==============================
train_size = int(len(series) * 0.8)

train = series[:train_size]
test = series[train_size:]


# ==============================
# 6. Fit ARIMA Model
# ==============================
# Example: ARIMA(1,1,1)
model = ARIMA(train, order=(1, 1, 1))
model_fit = model.fit()

print(model_fit.summary())


# ==============================
# 7. Forecast
# ==============================
forecast = model_fit.forecast(steps=len(test))

# Plot forecast
plt.figure()
plt.plot(train, label='Train')
plt.plot(test, label='Test')
plt.plot(test.index, forecast, label='Forecast')
plt.legend()
plt.title("ARIMA Forecast")
plt.show()


# ==============================
# 8. Evaluate Forecast Accuracy
# ==============================
mae = mean_absolute_error(test, forecast)
rmse = np.sqrt(mean_squared_error(test, forecast))

print("MAE:", mae)
print("RMSE:", rmse)


# ==============================
# 9. Residual Diagnostics
# ==============================
residuals = model_fit.resid

print("Residual Mean:", residuals.mean())

plt.figure()
plt.plot(residuals)
plt.title("Residuals")
plt.show()